In [2]:
import numpy as np

Doing matrix operations over integers, rather than a field.

In [3]:
def bezout(n: int, m: int) -> list[int]:
    ''' Given two integers n and m, find integers x, y such that
        x*n + y*m = gcd(n, m)
        Returns [x, y, gcd(n, m)] '''
    
    # Initialize: [x, y, value]
    x0, y0, r0 = 1, 0, n
    x1, y1, r1 = 0, 1, m
    
    while r1 != 0: # do euclidean algorithm
        q = r0 // r1
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
        r0, r1 = r1, r0 - q * r1
    
    return [x0, y0, r0]

In [4]:
A = np.array([[1, 2, 3], [3, 4, 5]]); M=A.copy()
M[:, [0,1]] = 2* M[:, 1], M[:, 0]
A[:, [0,1]]

array([[1, 2],
       [3, 4]])

In [5]:
def op(row_or_col : str, type : str, data: list, L, M, R):
    '''Perform either row or column operations to the integer matrix M.
     Modifyies L, M, R in place so that the product LMR remains unchanged '''
    if row_or_col == 'row':
        # row operation
        if type == 'swap': # swap two rows
            i, j = data
            M[[i, j]] = M[[j, i]]
            L[[i, j]] = L[[j, i]]
        elif type == 'negate':
            i = data
            M[i] *= -1
            L[:, i] *= -1
        elif type == 'add':
            # row_i |-> row_i + k* row_j
            i, j, k = data
            M[i] = M[i] + k * M[j]
            L[:, j] = L[:, j] -k * L[:, i]
        elif type in {'bez', 'bezout'}:
            ''' Given x*n + y*m = gcd(m, m) =:d , multiply by the matrix
                    [  x    y  ]
                    [-m/d  n/d ]
            applied to the submatrix of rows i and j and identity elsewhere'''
            #-----------------------------------------------------------------
            # subroutine to find the gcd and bezout coefficients
            def bezout(n: int, m: int) -> list[int]:
                ''' Given two integers n and m, find integers x, y such that
                x*n + y*m = gcd(n, m)
                Returns [x, y, gcd(n, m)]'''    
                # Initialize: [x, y, value]
                x0, y0, r0 = 1, 0, n
                x1, y1, r1 = 0, 1, m                
                while r1 != 0: # do euclidean algorithm
                    q = r0 // r1
                    x0, x1 = x1, x0 - q * x1
                    y0, y1 = y1, y0 - q * y1
                    r0, r1 = r1, r0 - q * r1                
                return [x0, y0, r0]
            #------------------------------------------------------------------
            i, j, n, m = data
            x, y, d = bezout(n, m)
            a, b = n // d, m // d

            M[[i, j]] = x * M[i] + y* M[j], -b * M[i] + a * M[j]
            L[:, i], L[:, j] = (a * L[:, i] + b * L[:, j], 
                                -y * L[:, i] + x * L[:, j] )
        else:
            print('Error: that operation not supported')
    elif row_or_col in {'col', 'column'}:
        # take transpose and do row operations 
        op('row', type, data, R.T, M.T, L.T)
        


In [11]:
L = R = np.eye(3, dtype='i')
M = np.random.randint(-5,5, (3,3))
L_, M_, R_ = L.copy(), M.copy(), R.copy()
op('row', 'bez', [0,2,4, 6], L_, M_, R_)
op('row', 'bez', [0,1,2, 7], L_, M_, R_)
op('col', 'add', [0,2,-3], L_, M_, R_)
op('col', 'bez', [1,2,2, 3], L_, M_, R_)
op('row', 'negate', 2, L_, M_ , R_)
print(f'L=\n{L},\nM=\n{M},\nR=\n{R},\nL@M@R=\n{L@M@R})\n\nL_=\n{L_},\nM_=\n{M_},\nR_=\n{R_},\nL_@M_@R_=\n{L_@M_@R_}')

L=
[[1 0 0]
 [0 1 0]
 [0 0 1]],
M=
[[-3 -3 -3]
 [-5 -4  3]
 [-3 -4  4]],
R=
[[1 0 0]
 [0 1 0]
 [0 0 1]],
L@M@R=
[[-3 -3 -3]
 [-5 -4  3]
 [-3 -4  4]])

L_=
[[ 4 -2  1]
 [ 7 -3  0]
 [ 6 -3  1]],
M_=
[[ 49 -17 -33]
 [119 -42 -83]
 [ 48 -16 -31]],
R_=
[[ 1  0  0]
 [ 9  2  3]
 [-3 -1 -1]],
L_@M_@R_=
[[-3 -3 -3]
 [-5 -4  3]
 [-3 -4  4]]


In [7]:
def setup_pivot(L, M, R, row=0) -> int:
    ''' Given array M, finds the first pivot column, swaps that row 
    to the top. Modifies the input arrays M and the array L keeps track
    of the row operations performed.
    '''
    # finds the pivot of the bottom right row x row submatrix
    n, m = M.shape
    for col in range(row, m):
        # Find pivot            
        pivot_row = None
        for other_row in range(row, n):
            if M[other_row, col] != 0:
                pivot_row = other_row
                break
        if pivot_row is not None:
            # Swap pivot row into position
            if pivot_row != row:
                op('row', 'swap', [row, pivot_row], L, M, R)
            return col  # return immediately when pivot is found
    return None # all zeros, so no pivot


L = R = np.eye(3, dtype='i')
M = np.array([[0, 0, 0], [4, 0, 0], [7, 0, 9]])
L_, M_, R_ = L.copy(), M.copy(), R.copy()
col = setup_pivot(L_, M_, R_, row=1)
L_, M_, R_, col

(array([[1, 0, 0],
        [0, 0, 1],
        [0, 1, 0]], dtype=int32),
 array([[0, 0, 0],
        [7, 0, 9],
        [4, 0, 0]]),
 array([[1, 0, 0],
        [0, 1, 0],
        [0, 0, 1]], dtype=int32),
 2)

In [8]:
def smith(matrix, order_divisors=False):
    '''Given a matrix M with integer coefficients, returns a triple:
    [L, D, R] of matrices with integer coefficients, where L and R are 
    invertible over Z, D is diagonal and LDR = M. If order_divisors, 
    then D is arranged so that a_j divides a_{j+1} along the diagonal.'''

    def row_reduce(matrix, pivots=False):
        ''' subroutine which does just the row operations. Returns
        [D, R] where R is invertible, D is reduced and M = DR'''
        M = np.array(matrix, dtype='i')
        n, m = M.shape
        L = np.identity(n, dtype='i') 


        # We will do row reduction, except multiplying row by const isn't 
        # allowed unles const = +- 1
        # will use euclidean algorithm to find invertible row operations
        def bezout(n: int, m: int) -> list[int]:
            ''' Given two integers n and m, find integers x, y such that
                x*n + y*m = gcd(n, m)
                Returns [x, y, gcd(n, m)] '''
            
            # Initialize: [x, y, value]
            x0, y0, r0 = 1, 0, n
            x1, y1, r1 = 0, 1, m
            
            while r1 != 0: # do euclidean algorithm
                q = r0 // r1
                x0, x1 = x1, x0 - q * x1
                y0, y1 = y1, y0 - q * y1
                r0, r1 = r1, r0 - q * r1
            
            return [x0, y0, r0]
        
        # Do row reduction and keep track of row operations with matrix L
        row = 0
        pivot_cols = []
        for col in range(m):
            # Find pivot            
            pivot_row = None
            for oth in range(row, n):
                if M[oth, col]:
                    pivot_row = oth
                    break
            if pivot_row is None:
                continue

            # Swap pivot row into position
            if pivot_row != row:
                M[[row, pivot_row]] = M[[pivot_row, row]]

                # Keep track with L matrix
                row_swap = np.identity(n, dtype='i')
                row_swap[pivot_row, pivot_row] = row_swap[row, row] = 0
                row_swap[pivot_row, row] = row_swap[row, pivot_row] = 1
                L @= row_swap # update L matrix

            # make the pivot positive
            if M[row, col] < 0:
                M[row] *= -1
                negate_row = np.identity(n, dtype='i')
                negate_row[row, row] = -1
                L @= negate_row

            # Eliminate other rows `oth` using bezout coefficients
            for oth in range(n):
                # if another row isn't a multiple of the pivot
                if M[oth, col] % M[row, col] != 0:
                    # perform invertible row operation to pass to gcd
                    piv, other = M[row, col], M[oth, col]
                    x, y, gcd = bezout(piv, other) # bezout coefficients
                    a, b = piv // gcd, other // gcd
                    M[row], M[oth] = x*M[row] + y*M[oth], a*M[oth] - b*M[row]
                    # now M[row, col] = gcd and M[oth, col] = 0

                    # encode this operation in the L matrix
                    bez_op = np.identity(n, dtype='i')
                    bez_op[row, row], bez_op[row, oth] = a, -y
                    bez_op[oth, row], bez_op[oth, oth] = b, x
                    # bez_op is determinant 1, so invertible over Z
                    L @= bez_op

            # now that we've made the row entries divisible by pivot,
            # we can finally eliminate by subtracting off pivot row
            for oth in range(n):
                if oth != row and M[oth, col] != 0:
                    # should be divisible by M[row, col] if above worked
                    q = M[oth, col] // M[row, col]
                    M[oth] -= q * M[row]

                    # encode in the R matrix
                    subtract_row = np.identity(n, dtype='i')
                    subtract_row[oth, row] = q
                    L @= subtract_row

            pivot_cols.append(col)
            row += 1
            if row == n:
                break
        return [L, M] if not pivots else [L, M, pivot_cols]

    L, M = row_reduce(matrix)
    # now do the columns by taking transpose
    col_reduce = row_reduce(M.T, pivots=True)
    R, D, pivots = col_reduce[0].T, col_reduce[1].T, col_reduce[2]
    return [L, D, R]



In [9]:
A = np.array([[4, 3, 2],[6, 5, 4], [10, 9, 8]])
B = np.array([[1, 1],[1, 1]])

In [10]:
A = np.random.randint(-2, 3, size=(4, 3))
#A = np.array([[4, 3, 2],[6, 5, 4], [10, 9, 8]])
L, D, R = smith(A)
A, L@D@R, L, D, R

(array([[ 0,  2, -2],
        [ 1,  1, -2],
        [ 0, -2, -1],
        [ 0,  1,  2]]),
 array([[ 0,  2, -2],
        [ 1,  1, -2],
        [ 0, -2, -1],
        [ 0,  1,  2]]),
 array([[-1,  2,  2,  0],
        [ 0,  1,  2,  0],
        [ 3, -2,  1,  0],
        [ 0,  1, -2,  1]], dtype=int32),
 array([[  6,   0,   0],
        [  0,   1,   0],
        [  0,   0,   1],
        [-15,   0,   0]]),
 array([[-1,  0,  0],
        [-7,  1,  0],
        [ 4,  0, -1]], dtype=int32))